In [1]:
!pip install -q -U transformers accelerate optimum-quanto

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.3 MB/s eta 0:00:00


In [2]:
import torch, time, gc, os, re
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA:", torch.cuda.is_available())

CUDA: True


In [3]:
import transformers
print(transformers.__version__)
from optimum.quanto import QuantizedModelForCausalLM  # just a sanity import check
print("quanto import OK")

5.15.0
quanto import OK


In [5]:
from datasets import load_dataset
import random
random.seed(42)

gsm8k = load_dataset("openai/gsm8k", "main", split="test")
gsm8k = gsm8k.shuffle(seed=42).select(range(200))

def extract_gsm8k_answer(answer_field):
    return answer_field.split("####")[-1].strip().replace(",", "")

gsm8k_questions = [
    {"question": ex["question"], "gold": extract_gsm8k_answer(ex["answer"])}
    for ex in gsm8k
]

mmlu = load_dataset("cais/mmlu", "all", split="test")
mmlu = mmlu.shuffle(seed=42).select(range(200))
letters = ["A", "B", "C", "D"]
mmlu_questions = []
for ex in mmlu:
    choices_str = "\n".join(f"{letters[i]}. {c}" for i, c in enumerate(ex["choices"]))
    mmlu_questions.append({
        "question": f"{ex['question']}\n{choices_str}",
        "gold": letters[ex["answer"]],
        "subject": ex["subject"],
    })
print(len(gsm8k_questions), len(mmlu_questions))

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

... (12 outputs left out)

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

200 200


In [6]:
GSM8K_PROMPT = (
    "Solve this math problem step by step, then give the final numeric answer "
    "on the last line as 'Answer: <number>'.\n\nProblem: {q}\n"
)
MMLU_PROMPT = (
    "Answer this multiple-choice question. Respond with only the letter "
    "(A, B, C, or D) on the last line as 'Answer: <letter>'.\n\nQuestion: {q}\n"
)

def extract_number(text):
    text = text.replace(",", "")
    matches = re.findall(r"Answer:\s*(-?\d+\.?\d*)", text)
    if matches:
        return matches[-1]
    nums = re.findall(r"-?\d+\.?\d*", text)
    return nums[-1] if nums else None

def extract_letter(text):
    matches = re.findall(r"Answer:\s*([ABCD])", text.upper())
    if matches:
        return matches[-1]
    matches = re.findall(r"\b([ABCD])\b", text.upper())
    return matches[-1] if matches else None

@torch.no_grad()
def run_benchmark(generate_fn, questions, task_type, max_new_tokens=512, limit=None):
    qs = questions[:limit] if limit else questions
    correct = 0; total_tokens = 0; total_time = 0.0; records = []
    for i, ex in enumerate(qs):
        prompt = GSM8K_PROMPT.format(q=ex["question"]) if task_type == "gsm8k" else MMLU_PROMPT.format(q=ex["question"])
        t0 = time.time()
        output_text, n_tokens = generate_fn(prompt, max_new_tokens)
        dt = time.time() - t0
        total_time += dt; total_tokens += n_tokens
        pred = extract_number(output_text) if task_type == "gsm8k" else extract_letter(output_text)
        gold = ex["gold"]
        is_correct = (pred is not None) and (str(pred).strip() == str(gold).strip())
        correct += int(is_correct)
        records.append({"question": ex["question"][:80], "gold": gold, "pred": pred, "correct": is_correct, "subject": ex.get("subject")})
        if (i + 1) % 10 == 0:
            print(f"  ...{i+1}/{len(qs)} done, running acc so far: {100*correct/(i+1):.1f}%")
    return {"accuracy_pct": 100*correct/len(qs), "tokens_per_sec": total_tokens/total_time if total_time>0 else 0, "n_questions": len(qs), "records": records}

def reset_memory():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

results_log = []
def log_result(variant_name, task_type, bench_result, peak_mem_gb, load_time_s):
    results_log.append({"variant": variant_name, "task": task_type, "accuracy_pct": bench_result["accuracy_pct"],
                         "tokens_per_sec": bench_result["tokens_per_sec"], "peak_mem_gb": peak_mem_gb,
                         "load_time_s": load_time_s, "n_questions": bench_result["n_questions"]})
    print(f"[{variant_name} | {task_type}] acc={bench_result['accuracy_pct']:.1f}%  tok/s={bench_result['tokens_per_sec']:.1f}  mem={peak_mem_gb:.2f}GB")

In [7]:
MODEL_ID = "microsoft/Phi-4-mini-instruct"
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

reset_memory()
t0 = time.time()
model_kvquant = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cuda")
load_time_kvquant = time.time() - t0

def make_kv_generate_fn(model, tok, nbits=4, backend="quanto"):
    def generate_fn(prompt, max_new_tokens):
        messages = [{"role": "user", "content": prompt}]
        inputs = tok.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)
        in_len = inputs["input_ids"].shape[1]
        cache = transformers.QuantizedCache(backend=backend, config=model.config, nbits=nbits)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tok.pad_token_id, past_key_values=cache,
            )
        text = tok.decode(out[0][in_len:], skip_special_tokens=True)
        n_new = out.shape[1] - in_len
        del cache, out, inputs
        gc.collect(); torch.cuda.empty_cache()
        return text, n_new
    return generate_fn

gen_kvquant = make_kv_generate_fn(model_kvquant, tokenizer, nbits=4, backend="quanto")
print("Loaded OK")

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Loaded OK


In [8]:
print(f"Before: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
prompt = GSM8K_PROMPT.format(q=gsm8k_questions[0]["question"])
text, n = gen_kvquant(prompt, 128)
print(f"After: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(text[:200])

Before: 7.15 GB


ValueError: `QuantizedCache` is only supported for models with only full attention layers. We found the following invalid layer types: {'sliding_attention'}